In [ ]:
#REPACKAGE MODEL

from sagemaker.huggingface import HuggingFace

role = "arn:aws:iam::954*******:role/SageMakerExecutionRole-LLMProject"

repackage_estimator_v2 = HuggingFace(
    entry_point="repackage.py",
    source_dir=".",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.49",
    pytorch_version="2.5",
    py_version="py311",
)

repackage_estimator_v2.fit({
    "model": "s3://sagemaker-ap-south-1-954*******/huggingface-pytorch-training-2026-08-29-06-51-06-814/output/model.tar.gz"
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-09-01-04-37-27-628


2026-09-01 04:38:12 Starting - Starting the training job
2026-09-01 04:38:12 Pending - Training job waiting for capacity...................................................
2026-09-01 04:46:54 Pending - Preparing the instances for training...
2026-09-01 04:47:18 Downloading - Downloading input data.........
2026-09-01 04:49:09 Downloading - Downloading the training image............
2026-09-01 04:51:10 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.163.01
Current installed NVIDIA driver version is 595.91.07
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-09-01 04:51:16,152 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-09-01 04:51:16,176 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons 

In [ ]:
# CREATE MODEL

import boto3

sm_client = boto3.client("sagemaker", region_name="ap-south-1")

role = "arn:aws:iam::954*******:role/SageMakerExecutionRole-LLMProject"
image_uri = "763104351884.dkr.ecr.ap-south-1.amazonaws.com/huggingface-pytorch-inference:2.6-transformers4.49-gpu-py312-cu124-ubuntu22.04"
model_data = "s3://sagemaker-ap-south-1-954*******:role/SageMakerExecutionRole-LLMProject"

sm_client.create_model(
    ModelName="phi3-llmproject-model-v4",
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": model_data,
        "Environment": {
            "HF_TRUST_REMOTE_CODE": "true",
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    },
    ExecutionRoleArn=role,
)
print("Model created.")

Model created.


In [20]:
sm_client.create_endpoint(
    EndpointName="phi3-llmproject-endpoint-v8",
    EndpointConfigName="phi3-llmproject-config-v6",
)
print("Endpoint creation request submitted.")

Endpoint creation request submitted.


In [ ]:
from sagemaker.huggingface import HuggingFace

role = "arn:aws:iam::954*******:role/SageMakerExecutionRole-LLMProject"

validate_estimator = HuggingFace(
    entry_point="validate_inference.py",
    source_dir=".",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.49",
    pytorch_version="2.5",
    py_version="py311",
)

validate_estimator.fit({
    "model": "s3://sagemaker-ap-south-1-954*******:role/SageMakerExecutionRole-LLMProject"
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-09-01-05-59-07-234


2026-09-01 05:59:53 Starting - Starting the training job
2026-09-01 05:59:53 Pending - Training job waiting for capacity............
2026-09-01 06:01:55 Pending - Preparing the instances for training...
2026-09-01 06:02:19 Downloading - Downloading input data............
2026-09-01 06:04:25 Downloading - Downloading the training image.........
2026-09-01 06:06:11 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.163.01
Current installed NVIDIA driver version is 595.91.07
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-09-01 06:06:18,402 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-09-01 06:06:18,425 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-09-01 06:06:18,434 sa

In [31]:
%%writefile inference.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache

if not hasattr(DynamicCache, "get_max_length"):
    DynamicCache.get_max_length = DynamicCache.get_max_cache_shape

def model_fn(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto",
    )
    return {"model": model, "tokenizer": tokenizer}

def predict_fn(data, model_dict):
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    prompt = data.get("inputs", "")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return {"generated_text": result}

Overwriting inference.py


In [ ]:
from sagemaker.huggingface import HuggingFace

role = "arn:aws:iam::954*******:role/SageMakerExecutionRole-LLMProject"

repackage_final = HuggingFace(
    entry_point="repackage.py",
    source_dir=".",
    instance_type="ml.g4dn.xlarge",   # <-- switched from g5.xlarge
    instance_count=1,
    role=role,
    transformers_version="4.49",
    pytorch_version="2.5",
    py_version="py311",
)

repackage_final.fit({
    "model": "s3://sagemaker-ap-south-1-954*******:role/SageMakerExecutionRole-LLMProject"
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-09-01-06-43-43-530


2026-09-01 06:44:19 Starting - Starting the training job
2026-09-01 06:44:19 Pending - Training job waiting for capacity...............
2026-09-01 06:46:53 Pending - Preparing the instances for training...
2026-09-01 06:47:20 Downloading - Downloading input data............
2026-09-01 06:49:16 Downloading - Downloading the training image..............bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.163.01
Current installed NVIDIA driver version is 595.91.07
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-09-01 06:51:48,396 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-09-01 06:51:48,420 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-09-01 06:51:48,430 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succe

In [ ]:
final_model_path = repackage_final.model_data
print(final_model_path)

In [ ]:
model_data_final = "s3://sagemaker-ap-south-1-954*******:role/SageMakerExecutionRole-LLMProject"  # from repackage_final.model_data

sm_client.create_model(
    ModelName="phi3-llmproject-model-final2",
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": model_data_final,
        "Environment": {
            "HF_TRUST_REMOTE_CODE": "true",
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    },
    ExecutionRoleArn=role,
)
print("Model created.")

sm_client.create_endpoint_config(
    EndpointConfigName="phi3-llmproject-config-final2",
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": "phi3-llmproject-model-final2",
        "InstanceType": "ml.g4dn.xlarge",
        "InitialInstanceCount": 1,
    }],
)
print("Endpoint config created.")

Model created.
Endpoint config created.


In [68]:
#CREATE THE  ENDPOINT

sm_client.create_endpoint(
    EndpointName="phi3-llmproject-endpoint-final",
    EndpointConfigName="phi3-llmproject-config-final2",
)
print("Endpoint creation request submitted.")

Endpoint creation request submitted.


In [69]:
#MONITORING ENDPOINT LOOP

import time

for i in range(20):
    response = sm_client.describe_endpoint(EndpointName="phi3-llmproject-endpoint-final")
    status = response["EndpointStatus"]
    print(f"Check {i+1}: {status}")
    if status == "InService":
        print("Ready!")
        break
    elif status == "Failed":
        print("FAILED:", response.get("FailureReason"))
        break
    time.sleep(20)

Check 1: Creating
Check 2: Creating
Check 3: Creating
Check 4: Creating
Check 5: Creating
Check 6: Creating
Check 7: Creating
Check 8: Creating
Check 9: Creating
Check 10: Creating
Check 11: Creating
Check 12: Creating
Check 13: Creating
Check 14: Creating
Check 15: Creating
Check 16: Creating
Check 17: Creating
Check 18: Creating
Check 19: Creating
Check 20: Creating


In [ ]:
#TESTING THE MODEL FROM ENDPOINT

import json
from botocore.config import Config

config = Config(read_timeout=30, connect_timeout=10)
runtime_client = boto3.client("sagemaker-runtime", region_name="ap-south-1", config=config)

payload = {
    "inputs": "### Instruction:\nWhat is the capital of France?\n\n### Response:\n"
}

try:
    response = runtime_client.invoke_endpoint(
        EndpointName="phi3-llmproject-endpoint-final",
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    result = json.loads(response["Body"].read().decode())
    print("MODEL RESPONSE:", result)
except Exception as e:
    print("Invoke failed or timed out:", e)

MODEL RESPONSE: {'generated_text': '### Instruction:\nWhat is the capital of France?\n\n### Response:\nThe capital of France is Paris. The city is located on the river Seine in the northern part of the country. Paris is the second largest city in France and serves as the main center for commerce, culture, art, fashion, cinema, and design. The city is also known for its history and architecture, with landmarks such as the Louvre, Eiffel Tower, Notre Dame Cathedral, and the Arc de Triomphe. Paris is also a UNESCO World Heritage Site'}


In [ ]:
#FOR DELETING THE ENDPOINT

sm_client.delete_endpoint(EndpointName="phi3-llmproject-endpoint-final")

print("Endpoint deleted.")

In [ ]:
#CREATING LAMBDA FUNCTION

import boto3

lambda_client = boto3.client("lambda", region_name="ap-south-1")

with open("lambda_function.zip", "rb") as f:
    zip_bytes = f.read()

lambda_role = "arn:aws:iam::954*******:role/LambdaExecutionRole-LLMProject"

response = lambda_client.create_function(
    FunctionName="phi3-llmproject-inference",
    Runtime="python3.12",
    Role=lambda_role,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": zip_bytes},
    Timeout=30,          # generous timeout since model inference can take a few seconds
    MemorySize=256,       
)

print("Lambda function created:", response["FunctionArn"])

In [49]:
import json

test_event = {
    "body": json.dumps({"prompt": "What is the capital of France?"})
}

response = lambda_client.invoke(
    FunctionName="phi3-llmproject-inference",
    InvocationType="RequestResponse",
    Payload=json.dumps(test_event),
)

result = json.loads(response["Payload"].read())
print(result)

{'statusCode': 200, 'headers': {'Content-Type': 'application/json'}, 'body': '{"response": "### Instruction:\\nWhat is the capital of France?\\n\\n### Response:\\nParis is the capital of France. It is located in the northern part of the country. The city is known for its art, architecture, fashion, and food. The Eiffel Tower and the Louvre Museum are popular tourist attractions. The city is also a political center, with the French government and many international organizations located there. Paris is a global city that is influential in the fields of culture, politics, and economics. The city is also known for its history and"}'}


In [ ]:
#CREATING API GATEWAY

apigw_client = boto3.client("apigateway", region_name="ap-south-1")

api_response = apigw_client.create_rest_api(
    name="phi3-llmproject-api",
    description="Serverless API for fine-tuned Phi-3 LLM inference",
)

api_id = api_response["id"]
print("API created:", api_id)

API created: umt5emd863


In [ ]:
#CREATING RESOURCE AND METHOD

resources = apigw_client.get_resources(restApiId=api_id)
root_id = resources["items"][0]["id"]
print("Root resource ID:", root_id)

Root resource ID: 8gcckoo7yf


In [54]:
resource_response = apigw_client.create_resource(
    restApiId=api_id,
    parentId=root_id,
    pathPart="generate",
)

resource_id = resource_response["id"]
print("Resource created:", resource_id)

Resource created: u95yjl


In [55]:
apigw_client.put_method(
    restApiId="umt5emd863",
    resourceId="u95yjl",
    httpMethod="POST",
    authorizationType="NONE",  # no auth for portfolio/demo purposes; could add API keys later
)
print("POST method created.")

POST method created.


In [ ]:
#LAMBDA INTEGRATION WITH API GATEWAY


lambda_arn = "arn:aws:lambda:ap-south-1:954*******:function:phi3-llmproject-inference"
uri = f"arn:aws:apigateway:ap-south-1:lambda:path/2015-03-31/functions/{lambda_arn}/invocations"

apigw_client.put_integration(
    restApiId="umt5emd863",
    resourceId="u95yjl",
    httpMethod="POST",
    type="AWS_PROXY",
    integrationHttpMethod="POST",
    uri=uri,
)
print("Integration created.")

Integration created.


In [ ]:
#Grant API Gateway permission to invoke Lambda


lambda_client.add_permission(
    FunctionName="phi3-llmproject-inference",
    StatementId="apigateway-invoke",
    Action="lambda:InvokeFunction",
    Principal="apigateway.amazonaws.com",
    SourceArn=f"arn:aws:execute-api:ap-south-1:954*******:umt5emd863/*/POST/generate",
)
print("Permission granted.")

Permission granted.


In [60]:
#DEPLOYING API GATEWAY

deployment = apigw_client.create_deployment(
    restApiId="umt5emd863",
    stageName="prod",
)
print("API deployed to 'prod' stage.")

API deployed to 'prod' stage.


In [61]:
api_url = f"https://umt5emd863.execute-api.ap-south-1.amazonaws.com/prod/generate"
print("Your API endpoint:", api_url)

Your API endpoint: https://umt5emd863.execute-api.ap-south-1.amazonaws.com/prod/generate


In [62]:
#CREATING DYNAMODB TABLE FOR LOGGING and MONITORING

dynamodb_client = boto3.client("dynamodb", region_name="ap-south-1")

dynamodb_client.create_table(
    TableName="phi3-llmproject-logs",
    KeySchema=[
        {"AttributeName": "request_id", "KeyType": "HASH"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "request_id", "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",  # no fixed cost, pay only per read/write — ideal for low-volume portfolio use
)
print("DynamoDB table created.")

DynamoDB table created.


In [70]:
import requests

api_url = "https://umt5emd863.execute-api.ap-south-1.amazonaws.com/prod/generate"

payload = {"prompt": "What is the capital of France?"}

response = requests.post(api_url, json=payload, timeout=30)
print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'response': '### Instruction:\nWhat is the capital of France?\n\n### Response:\nThe capital of France is Paris. Paris is also known as the "city of lights" and it\'s a very popular tourist destination. It\'s also known for its art and culture. Some of the most famous landmarks in Paris include the Eiffel Tower, Notre Dame Cathedral, the Louvre Museum, and the Arc de Triomphe. \n\nParis is also the largest city in France and it\'s one of the most populous cities in Europe'}
